In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score, accuracy_score
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader


In [4]:
# load small subset for demo
df = pd.read_csv("sample_data/train.csv", nrows=50000)

df.head()


,id,target,ps_ind_01,ps_ind_02_cat,ps_ind_03,ps_ind_04_cat,ps_ind_05_cat,ps_ind_06_bin,ps_ind_07_bin,ps_ind_08_bin,...,ps_calc_11,ps_calc_12,ps_calc_13,ps_calc_14,ps_calc_15_bin,ps_calc_16_bin,ps_calc_17_bin,ps_calc_18_bin,ps_calc_19_bin,ps_calc_20_bin
0,7,0,2,2,5,1,0,0,1,0,...,9,1,5,8,0,1,1,0,0,1
1,9,0,1,1,7,0,0,0,0,1,...,3,1,1,9,0,1,1,0,1,0
2,13,0,5,4,9,1,0,0,0,1,...,4,2,7,7,0,1,1,0,1,0
3,16,0,0,1,2,0,0,1,0,0,...,2,2,4,9,0,0,0,0,0,0
4,17,0,0,2,0,1,0,1,0,0,...,3,1,1,3,0,0,0,1,1,0


In [5]:
df['target'].value_counts(normalize=True)


,proportion
target,
0,0.96334
1,0.03666


In [6]:
df.describe().T.head()


,count,mean,std,min,25%,50%,75%,max
id,50000.0,62712.28968,36121.857807,7.0,31443.0,62519.5,93983.25,125414.0
target,50000.0,0.03666,0.187928,0.0,0.0,0.0,0.00,1.0
ps_ind_01,50000.0,1.89934,1.986335,0.0,0.0,1.0,3.00,7.0
ps_ind_02_cat,50000.0,1.35362,0.658378,-1.0,1.0,1.0,2.00,4.0
ps_ind_03,50000.0,4.41218,2.695194,0.0,2.0,4.0,6.00,11.0


In [7]:
(df == -1).sum().sort_values(ascending=False).head()


,0
ps_car_03_cat,34464
ps_car_05_cat,22236
ps_reg_03,9138
ps_car_14,3586
ps_car_07_cat,902


In [8]:
df = df.replace(-1, np.nan)
df = df.fillna(df.median())


In [9]:
X = df.drop(['id', 'target'], axis=1)
y = df['target']


In [10]:
X_train, X_valid, y_train, y_valid = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)


In [11]:
scaler = StandardScaler()

X_train_scaled = scaler.fit_transform(X_train)
X_valid_scaled = scaler.transform(X_valid)

X_train_scaled = pd.DataFrame(X_train_scaled, columns=X_train.columns)
X_valid_scaled = pd.DataFrame(X_valid_scaled, columns=X_valid.columns)


logistic regression

In [12]:
from sklearn.linear_model import LogisticRegression

lr = LogisticRegression(max_iter=200)
lr.fit(X_train_scaled, y_train)

pred_lr = lr.predict_proba(X_valid_scaled)[:,1]
print("LR AUC:", roc_auc_score(y_valid, pred_lr))


LR AUC: 0.631955434755245


tree

In [13]:
from sklearn.tree import DecisionTreeClassifier

dt = DecisionTreeClassifier(max_depth=6, class_weight='balanced')
dt.fit(X_train_scaled, y_train)

pred_dt = dt.predict_proba(X_valid_scaled)[:,1]
print("DT AUC:", roc_auc_score(y_valid, pred_dt))


DT AUC: 0.572115861942556


MLP Dataset and Loader

In [14]:
class TabDataset(Dataset):
    def __init__(self, X, y):
        self.X = torch.tensor(X.values, dtype=torch.float32)
        self.y = torch.tensor(y.values, dtype=torch.float32)
    def __len__(self):
        return len(self.X)
    def __getitem__(self, idx):
        return self.X[idx], self.y[idx]

train_ds = TabDataset(X_train_scaled, y_train)
valid_ds = TabDataset(X_valid_scaled, y_valid)

train_loader = DataLoader(train_ds, batch_size=512, shuffle=True)
valid_loader = DataLoader(valid_ds, batch_size=512)


MLP model

In [15]:
class MLP(nn.Module):
    def __init__(self, input_dim):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(input_dim, 128),
            nn.ReLU(),
            nn.Dropout(0.3),

            nn.Linear(128, 64),
            nn.ReLU(),
            nn.Dropout(0.3),

            nn.Linear(64, 1)
        )
    def forward(self, x):
        return self.net(x).squeeze(1)

model = MLP(X_train_scaled.shape[1])
criterion = nn.BCEWithLogitsLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=1e-4)


train and eval loop

In [16]:
def train_one_epoch(model, loader, criterion, optimizer):
    model.train()
    total = 0
    for Xb, yb in loader:
        optimizer.zero_grad()
        out = model(Xb)
        loss = criterion(out, yb)
        loss.backward()
        optimizer.step()
        total += loss.item()
    return total / len(loader)

def evaluate(model, loader):
    model.eval()
    preds = []
    ys = []
    with torch.no_grad():
        for Xb, yb in loader:
            out = model(Xb)
            preds.append(torch.sigmoid(out).numpy())
            ys.append(yb.numpy())
    preds = np.concatenate(preds)
    ys = np.concatenate(ys)
    return roc_auc_score(ys, preds)


train model

In [17]:
for epoch in range(5):
    loss = train_one_epoch(model, train_loader, criterion, optimizer)
    auc = evaluate(model, valid_loader)
    print(f"Epoch {epoch+1} | Loss={loss:.4f} | Val AUC={auc:.4f}")


Epoch 1 | Loss=0.5460 | Val AUC=0.4647
Epoch 2 | Loss=0.3447 | Val AUC=0.4666
Epoch 3 | Loss=0.2219 | Val AUC=0.4909
Epoch 4 | Loss=0.1781 | Val AUC=0.5195
Epoch 5 | Loss=0.1657 | Val AUC=0.5447


AUC evaluation

In [18]:
print("LR AUC:", roc_auc_score(y_valid, pred_lr))
print("DT AUC:", roc_auc_score(y_valid, pred_dt))
print("MLP AUC (5 epochs):", evaluate(model, valid_loader))


LR AUC: 0.631955434755245
DT AUC: 0.572115861942556
MLP AUC (5 epochs): 0.5446547701178199
